# MobileFaceNet

## Add New Embeddings

In [ ]:
!pip3 install torch torchvision torchaudio

In [51]:
import os

# cd to /content/drive/MyDrive/Face_Dataset
%cd Face_Dataset

# print(os.getcwd())

[Errno 2] No such file or directory: 'Face_Dataset'
/Users/kyle/repos/named-ai/data-preprocessing/Face_Dataset/MobileFaceNet


In [ ]:
# Step 1 Clone the MobileFaceNet repository
!git clone https://github.com/foamliu/MobileFaceNet.git

Cloning into 'MobileFaceNet'...
remote: Enumerating objects: 589, done.
remote: Counting objects: 100% (206/206), done.
remote: Compressing objects: 100% (22/22), done.
remote: Total 589 (delta 186), reused 184 (delta 184), pack-reused 383 (from 1)
Receiving objects: 100% (589/589), 5.95 MiB | 9.34 MiB/s, done.
Resolving deltas: 100% (224/224), done.
Updating files: 100% (236/236), done.


In [6]:
# Step 2: Change directory into the repo
%cd MobileFaceNet

# Step 3: Make a directory for weights
!mkdir -p weights

# Step 4: Change into weights directory
%cd weights

# Step 5: Download the pretrained model
!wget https://github.com/foamliu/MobileFaceNet/releases/download/v1.0/mobilefacenet.pt

/Users/kyle/repos/named-ai/data-preprocessing/Face_Dataset/MobileFaceNet
/Users/kyle/repos/named-ai/data-preprocessing/Face_Dataset/MobileFaceNet/weights
--2025-09-10 21:41:32--  https://github.com/foamliu/MobileFaceNet/releases/download/v1.0/mobilefacenet.pt
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/220410268/00c4f100-0d46-11ea-8422-dff05b11ac5c?sp=r&sv=2018-11-09&sr=b&spr=https&se=2025-09-10T14%3A16%3A26Z&rscd=attachment%3B+filename%3Dmobilefacenet.pt&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2025-09-10T13%3A15%3A41Z&ske=2025-09-10T14%3A16%3A26Z&sks=b&skv=2018-11-09&sig=ahfTst%2BwehrUlu4JsS%2BY1z5nPKuenybXKbH5gn99eiQ%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYX

In [7]:
%cd MobileFaceNet

[Errno 2] No such file or directory: 'MobileFaceNet'
/Users/kyle/repos/named-ai/data-preprocessing/Face_Dataset/MobileFaceNet/weights


In [52]:
from mobilefacenet import MobileFaceNet
import torch
import time

# Load model
filename = 'weights/mobilefacenet.pt'
print(f'Loading {filename}...')
start = time.time()
model = MobileFaceNet()
model.load_state_dict(torch.load(filename, map_location=torch.device('cpu')))
model.eval()
print('Elapsed {:.2f} sec'.format(time.time() - start))


Loading weights/mobilefacenet.pt...
Elapsed 0.09 sec


In [53]:
from PIL import Image
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.Resize((112, 112)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

def get_embedding(image_path):
    img = Image.open(image_path).convert('RGB')
    input_tensor = transform(img).unsqueeze(0)  # Shape: (1, 3, 112, 112)
    with torch.no_grad():
        embedding = model(input_tensor)
        return embedding.squeeze()  # Shape: (128,)


In [54]:
face_db = {}
face_db['kyle'] = get_embedding('my_images/kyle_183.jpg')
face_db['Tom Cruise'] = get_embedding('my_images/Tom Cruise_12.jpg')

face_db.keys()

dict_keys(['kyle', 'Tom Cruise'])

In [68]:
# test_embedding = get_embedding('my_images/kyle_156.jpg')
# test_embedding = get_embedding('../processed_whole_face_dataset/val/kyle/kyle_161.jpg')
# test_embedding = get_embedding('../processed_whole_face_dataset/val/Zac Efron/Zac Efron_2.jpg')

test_embedding = get_embedding('../processed_whole_face_dataset/val/Tom Cruise/Tom Cruise_17.jpg')

In [56]:
from torch.nn.functional import cosine_similarity

def recognize_face(test_embedding, face_db, threshold=0.6):
    max_sim = 0
    identity = "Unknown"
    for name, db_embedding in face_db.items():
        sim = cosine_similarity(test_embedding.unsqueeze(0), db_embedding.unsqueeze(0))
        if sim.item() > max_sim and sim.item() > threshold:
            max_sim = sim.item()
            identity = name
    return identity, max_sim


In [69]:
# recognize face

identity, confidence = recognize_face(test_embedding, face_db)
print(f"Identified as: {identity} (Confidence: {confidence:.2f})")

Identified as: Tom Cruise (Confidence: 0.77)
